In [ ]:
# import required packages
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from matplotlib_venn import venn2
sns.set(style="whitegrid")
from docplex.mp.model import Model



In [ ]:
# Data Functions
def get_sp100_tickers():
    try:
        url = 'https://en.wikipedia.org/wiki/S%26P_100'
        tickers = pd.read_html(url)[2]['Symbol'].tolist()
        return [t.replace('.', '-') for t in tickers]
    except Exception:
        return ['AAPL', 'MSFT', 'AMZN', 'GOOG', 'GOOGL', 'META', 'TSLA', 'JNJ', 'JPM', 'V', 'PG', 'NVDA', 'HD', 'MA', 'DIS']

def download_data(tickers, start, end):
    data = yf.download(tickers, start=start, end=end, progress=False)
    close_prices = data['Close'].dropna(axis=1)
    returns = close_prices.pct_change().dropna()
    benchmark = returns.mean(axis=1)
    return returns, benchmark


In [ ]:
# Evaluation metrics
def print_portfolio_summary(name, weights, correlation, tracking_error, solve_time):
    print(f"\n=== Optimal Portfolio Using {name} ===")
    print("Selected Assets:")
    print(weights.sort_values(ascending=False).rename("Weight"))
    print(f"\nCorrelation: {correlation:.4f}")
    print(f"Tracking Error: {tracking_error:.6f}")
    print(f"Solve Time: {solve_time:.2f} seconds")

def evaluate_portfolio(returns, benchmark, weights):
    port_returns = returns[weights.index] @ weights
    corr = np.corrcoef(port_returns, benchmark)[0,1]
    te = np.sqrt(np.mean((port_returns - benchmark)**2))
    return port_returns, corr, te

def periodic_metrics(returns, benchmark, weights, label, periods):
    tracking_errors = {}
    correlations = {}

    for name, length in periods.items():
        period_returns = returns.iloc[-length:]
        period_benchmark = benchmark.iloc[-length:]

        port_returns = period_returns[weights.index].dot(weights)
        te = np.std(port_returns - period_benchmark)
        corr = port_returns.corr(period_benchmark)

        tracking_errors[name] = te
        correlations[name] = corr

        print(f"{label} - {name}: Tracking Error = {te:.6f}, Correlation = {corr:.4f}")

    return tracking_errors, correlations



In [ ]:
# MIQP 
def solve_miqp(returns, benchmark, q=10, time_limit=60):
    model = Model(name="MIQP")
    T, n = returns.shape
    assets = returns.columns
    x = model.continuous_var_list(n, name="x")
    y = model.binary_var_list(n, name="y")

    # Residuals setup
    residuals = model.sum(
        (model.sum(returns.values[t, i] * x[i] for i in range(n)) - benchmark.values[t]) ** 2
        for t in range(T)
    )
    model.minimize(residuals)

    # Constraints
    model.add_constraint(model.sum(x[i] for i in range(n)) == 1)
    model.add_constraint(model.sum(y[i] for i in range(n)) == q)
    for i in range(n):
        model.add_constraint(x[i] <= y[i])
        model.add_constraint(x[i] >= 0)

    # Solve and time it
    start_time = time.time()
    solution = model.solve(time_limit=time_limit)
    solve_time = time.time() - start_time

    if solution:
        weights = np.array([solution[x[i]] for i in range(n)])
        selected = weights > 1e-5
        weights = pd.Series(weights[selected], index=assets[selected])
        weights /= weights.sum()

        # Compute performance metrics
        selected_assets = assets[selected]
        portfolio_returns = returns[selected_assets].dot(weights)
        benchmark_returns = benchmark

        correlation = portfolio_returns.corr(benchmark_returns)
        tracking_error = np.std(portfolio_returns - benchmark_returns)

        return {
            'weights': weights,
            'correlation': correlation,
            'tracking_error': tracking_error,
            'solve_time': solve_time
        }
    else:
        return None


In [ ]:
# PSO

def solve_pso(returns, benchmark, q=10, particles=30, iterations=100):
    import time
    n = returns.shape[1]
    assets = returns.columns
    particle_positions = np.random.rand(particles, n)
    velocities = np.random.randn(particles, n) * 0.1
    p_best_pos = particle_positions.copy()
    p_best_scores = [fitness(p, returns, benchmark, q) for p in particle_positions]
    g_best = p_best_pos[np.argmin(p_best_scores)]

    def select_q(w):
        return np.argsort(w)[-q:]

    start = time.time()  # Start timing

    for _ in range(iterations):
        for i in range(particles):
            idx = select_q(particle_positions[i])
            w = np.zeros(n)
            w[idx] = 1 / q
            score = fitness(w, returns, benchmark, q)

            if score < p_best_scores[i]:
                p_best_pos[i] = particle_positions[i]
                p_best_scores[i] = score

                if score < fitness(g_best, returns, benchmark, q):
                    g_best = particle_positions[i]

            # Update velocity and position
            velocities[i] = (
                0.7 * velocities[i]
                + 1.5 * np.random.rand() * (p_best_pos[i] - particle_positions[i])
                + 1.5 * np.random.rand() * (g_best - particle_positions[i])
            )
            particle_positions[i] += velocities[i]
            particle_positions[i] = np.clip(particle_positions[i], 0, 1)

    solve_time = time.time() - start  # Stop timing

    # Final weights from best solution
    idx = select_q(g_best)
    weights = np.zeros(n)
    weights[idx] = 1 / q
    weights = pd.Series(weights[weights > 0], index=assets[weights > 0])

    # Portfolio metrics
    port_returns = returns[weights.index].dot(weights)
    correlation = port_returns.corr(benchmark)
    tracking_error = np.std(port_returns - benchmark)

    return {
        'weights': weights,
        'correlation': correlation,
        'tracking_error': tracking_error,
        'solve_time': solve_time
    }

def fitness(weights, returns, benchmark, q):
    w = np.array(weights)
    if np.count_nonzero(w) != q:
        return np.inf
    selected = returns.columns[w > 0]
    port_r = returns[selected].dot(w[w > 0])
    return np.sqrt(np.mean((port_r - benchmark)**2))



In [ ]:
# Main function
if __name__ == "__main__":
    tickers = get_sp100_tickers()
    returns, benchmark = download_data(tickers, "2023-01-01", "2024-01-01")
    q = 10
    periods = {'3mo': 63, '6mo': 126, '9mo': 189, '1y': len(returns)}

    # MIQP
    miqp_result = solve_miqp(returns, benchmark, q)

    if miqp_result:
        # Print metrics
        miqp_tracking_errors, miqp_correlations = periodic_metrics(returns, benchmark, miqp_result['weights'], "MIQP", periods)

        # Optional: If you're not using evaluate_portfolio anymore, skip this
        miqp_returns, miqp_corr, miqp_te = evaluate_portfolio(returns, benchmark, miqp_result['weights'])

        # Portfolio summary
        print_portfolio_summary(
            "MIQP",
            miqp_result['weights'],
            miqp_result['correlation'],
            miqp_result['tracking_error'],
            miqp_result['solve_time']
        )
        print("/n/n")
    else:
        print("MIQP failed to find a solution.")

    # PSO
    pso_result = solve_pso(returns, benchmark, q)

    if pso_result:
        pso_tracking_errors, pso_correlations = periodic_metrics(returns, benchmark, pso_result['weights'], "PSO", periods)

        print_portfolio_summary(
            "PSO",
            pso_result['weights'],
            pso_result['correlation'],
            pso_result['tracking_error'],
            pso_result['solve_time']
        )
    else:
        print("PSO failed to find a solution.")



### Results Interpretation
Metric	MIQP	PSO
1Y Correlation	0.9697	0.8911
1Y Tracking Error	0.0020	0.0039
Solve Time	60.08 sec	3.19 sec
Portfolio Diversity	Balanced, uneven weights	Even 10% across 10 assets
🧠 Interpretation:
✅ 1. Accuracy of Benchmark Tracking
MIQP wins clearly here.

It consistently maintains higher correlation and lower tracking error across all periods.

MIQP's returns follow the benchmark more closely — ideal for passive replication or benchmark tracking strategies.

📈 2. Risk & Precision
Tracking Error (TE) tells us how volatile the difference is between the portfolio and the benchmark.

MIQP TE is about half of PSO’s, meaning it delivers a smoother and more predictable tracking.

⚖️ 3. Weight Optimization
MIQP applies variable weighting, giving higher weight to stronger contributors (e.g., BRK-B, PEP).

PSO applies a uniform 10% allocation, likely due to its design — it finds a feasible solution quickly, but lacks refined allocation logic.

🕒 4. Speed
PSO solves the problem in ~3 seconds, while MIQP takes ~60 seconds.

If you're working with real-time or massive datasets and need just a decent solution fast, PSO is useful.

For high accuracy and robust tracking, MIQP is worth the extra time.

🏁 Final Verdict:
✅ If your priority is accuracy, robustness, and closer benchmark replication — MIQP is the better performer.

⚡️ If you’re okay with moderate tracking and want fast approximations — PSO is acceptable.

## Visualisations of results

In [ ]:
def plot_cumulative_returns(returns, benchmark, miqp_weights, pso_weights):
    import matplotlib.pyplot as plt

    miqp_cum = (1 + returns[miqp_weights.index].dot(miqp_weights)).cumprod()
    pso_cum = (1 + returns[pso_weights.index].dot(pso_weights)).cumprod()
    bench_cum = (1 + benchmark).cumprod()

    plt.figure(figsize=(12, 6))
    plt.plot(miqp_cum, label="MIQP", linewidth=2)
    plt.plot(pso_cum, label="PSO", linewidth=2)
    plt.plot(bench_cum, label="Benchmark", linestyle='--')
    plt.title("Cumulative Returns: MIQP vs PSO vs Benchmark")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('1st_chart.png')
    plt.show()

plot_cumulative_returns(returns, benchmark, miqp_result['weights'], pso_result['weights'])   

In [ ]:

if miqp_result and pso_result:
    miqp_assets = set(miqp_result['weights'].index)
    pso_assets = set(pso_result['weights'].index)

    overlap = miqp_assets & pso_assets
    only_miqp = miqp_assets - pso_assets
    only_pso = pso_assets - miqp_assets

    print("Asset Overlap Analysis:")
    print(f"MIQP Assets     : {sorted(miqp_assets)}")
    print(f"PSO Assets      : {sorted(pso_assets)}")
    print(f"Overlap ({len(overlap)} assets): {sorted(overlap)}")
    print(f"Only in MIQP    : {sorted(only_miqp)}")
    print(f"Only in PSO     : {sorted(only_pso)}")

    # Plot Venn Diagram
    plt.figure(figsize=(8, 6))
    venn2([miqp_assets, pso_assets], set_labels=('MIQP', 'PSO'), set_colors=('dodgerblue', 'tomato'))
    plt.title("Asset Selection Overlap: MIQP vs PSO")
    plt.savefig('2nd_chart.png')
    plt.show()


In [ ]:
def plot_correlation_vs_te(miqp_result, pso_result):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(6, 6))
    plt.scatter(miqp_result['tracking_error'], miqp_result['correlation'], label='MIQP', s=100, c='skyblue')
    plt.scatter(pso_result['tracking_error'], pso_result['correlation'], label='PSO', s=100, c='lightgreen')
    
    plt.xlabel("Tracking Error")
    plt.ylabel("Correlation with Benchmark")
    plt.title("Performance Comparison: MIQP vs PSO")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('3rd_chart.png')
    plt.show()
plot_correlation_vs_te(miqp_result, pso_result)

In [ ]:

def plot_comparison(miqp_te, pso_te, miqp_corr, pso_corr):
    periods = list(miqp_te.keys())
    x = np.arange(len(periods))  # the label locations
    width = 0.35  # the width of the bars

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Tracking Error Comparison
    ax1.bar(x - width/2, [miqp_te[p] for p in periods], width, label='MIQP', color='skyblue')
    ax1.bar(x + width/2, [pso_te[p] for p in periods], width, label='PSO', color='salmon')
    ax1.set_ylabel('Tracking Error')
    ax1.set_title('Tracking Error Comparison by Period')
    ax1.legend()
    ax1.grid(True, axis='y', linestyle='--', alpha=0.7)

    # Correlation Comparison
    ax2.bar(x - width/2, [miqp_corr[p] for p in periods], width, label='MIQP', color='skyblue')
    ax2.bar(x + width/2, [pso_corr[p] for p in periods], width, label='PSO', color='salmon')
    ax2.set_ylabel('Correlation with Benchmark')
    ax2.set_title('Correlation Comparison by Period')
    ax2.set_xticks(x)
    ax2.set_xticklabels(periods)
    ax2.legend()
    ax2.grid(True, axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('4th_chart.png')
    plt.show()
    
plot_comparison(
    miqp_te=miqp_tracking_errors,
    pso_te=pso_tracking_errors,
    miqp_corr=miqp_correlations,
    pso_corr=pso_correlations
)
